# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from lightgbm import LGBMClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_logit.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_logit.parquet')

In [4]:
X_train.head()

,lgbm_logit,cat_logit,xgb_logit,hist_logit,lg_logit,knn_logit,ridge_logit,voting_lgbm_cat_xgb_hist_logit,voting_lg_ridge_logit,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit
0,1.402666,2.109990,2.149899,2.030346,1.440273,2.064999,0.500714,1.887848,0.925326,1.226844
1,0.616425,1.595299,1.584592,1.262993,0.799597,0.079907,0.335105,1.222563,0.561368,0.371762
2,0.023615,1.748857,1.106745,1.146720,0.235157,-1.325798,0.136991,0.929687,0.186260,0.066618
3,-6.367775,-5.125195,-4.946636,-4.912857,-6.083520,-34.538776,-1.247050,-5.198584,-2.069743,-6.799934
4,-0.438053,0.263538,-0.043875,-0.068158,-0.346862,-0.861416,-0.094611,-0.070366,-0.220629,-0.878595


In [5]:
X_test.head()

,lgbm_logit,cat_logit,xgb_logit,hist_logit,lg_logit,knn_logit,ridge_logit,voting_lgbm_cat_xgb_hist_logit,voting_lg_ridge_logit,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit
0,-5.379032,-3.409957,-3.759100,-3.888929,-5.042109,-34.538776,-1.272709,-3.899528,-34.538776,-5.055490
1,-5.795381,-4.426299,-4.187473,-3.950213,-2.684867,-2.019087,-0.626960,-4.396275,-34.538776,-5.602505
2,-5.574937,-4.441487,-4.232580,-4.454119,-5.395432,-34.538776,-1.335019,-4.564839,-34.538776,-5.656104
3,-1.635113,0.176618,0.347033,-0.058463,0.644945,-2.198160,0.190133,-0.222683,-0.304675,-1.163384
4,1.920299,3.398116,3.163860,3.161400,1.661579,34.539576,0.535153,2.745303,0.794179,2.146481


# Machine Learning

In [8]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "alpha": trial.suggest_float("alpha", 1e-4, 100, log=True),
        "solver": trial.suggest_categorical("solver", ["auto", "svd", "cholesky", "lsqr", "sag", "sparse_cg"]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = RidgeClassifier(**params)
        model.fit(X_train, y_train)

        scores = model.decision_function(X_valid)

        auc = roc_auc_score(y_valid, scores)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-19 17:05:37,698] A new study created in memory with name: no-name-76468938-6fa8-4e5c-b7b5-66563c10a04a
Best trial: 1. Best value: 0.946357:   0%|▍                                                                                                                 | 2/500 [00:04<15:49,  1.91s/it]

[I 2026-05-19 17:05:42,019] Trial 4 finished with value: 0.9455195507186627 and parameters: {'alpha': 0.000651770929950179, 'solver': 'auto', 'class_weight': None, 'fit_intercept': True, 'tol': 0.0007736998869195268}. Best is trial 4 with value: 0.9455195507186627.
[I 2026-05-19 17:05:42,210] Trial 1 finished with value: 0.9463568010465278 and parameters: {'alpha': 38.82245847076902, 'solver': 'svd', 'class_weight': None, 'fit_intercept': False, 'tol': 2.1008787945233375e-05}. Best is trial 1 with value: 0.9463568010465278.


Best trial: 3. Best value: 0.94831:   1%|▋                                                                                                                  | 3/500 [00:04<09:23,  1.13s/it]

[I 2026-05-19 17:05:42,364] Trial 3 finished with value: 0.9483104189392433 and parameters: {'alpha': 0.06914292247230315, 'solver': 'auto', 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.00659654833575302}. Best is trial 3 with value: 0.9483104189392433.
[I 2026-05-19 17:05:42,379] Trial 7 finished with value: 0.948310410643038 and parameters: {'alpha': 0.05636761295240379, 'solver': 'auto', 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 8.066442615031282e-05}. Best is trial 3 with value: 0.9483104189392433.


Best trial: 0. Best value: 0.949515:   1%|█▏                                                                                                                | 5/500 [00:05<05:06,  1.61it/s]

...

[I 2026-05-19 17:08:51,519] Trial 495 pruned. 
[I 2026-05-19 17:08:51,571] Trial 496 finished with value: 0.9490494171131403 and parameters: {'alpha': 99.33230567039884, 'solver': 'lsqr', 'class_weight': None, 'fit_intercept': True, 'tol': 0.006095921855944322}. Best is trial 444 with value: 0.9495151579420383.


Best trial: 444. Best value: 0.949515: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 498/500 [03:14<00:00,  6.31it/s]

[I 2026-05-19 17:08:51,747] Trial 494 finished with value: 0.9490494167878112 and parameters: {'alpha': 99.12278857853536, 'solver': 'lsqr', 'class_weight': None, 'fit_intercept': True, 'tol': 0.008356714259022053}. Best is trial 444 with value: 0.9495151579420383.
[I 2026-05-19 17:08:51,760] Trial 497 finished with value: 0.9486953295940206 and parameters: {'alpha': 77.99050146057263, 'solver': 'lsqr', 'class_weight': None, 'fit_intercept': False, 'tol': 0.008623903246598915}. Best is trial 444 with value: 0.9495151579420383.
[I 2026-05-19 17:08:51,863] Trial 499 finished with value: 0.949049410118258 and parameters: {'alpha': 76.14842486963971, 'solver': 'lsqr', 'class_weight': None, 'fit_intercept': True, 'tol': 0.009762865078653421}. Best is trial 444 with value: 0.9495151579420383.
[I 2026-05-19 17:08:51,926] Trial 498 finished with value: 0.9490494093049145 and parameters: {'alpha': 74.35165459155428, 'solver': 'lsqr', 'class_weight': None, 'fit_intercept': True, 'tol': 0.0085997

Best trial: 444. Best value: 0.949515: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [03:18<00:00,  2.52it/s]

[I 2026-05-19 17:08:55,721] Trial 460 pruned. 
Best trial score:
0.9495151579420383

Best params:
{'alpha': 99.36883070481686, 'solver': 'lsqr', 'class_weight': None, 'fit_intercept': False, 'tol': 0.006964629514169135}


In [9]:
stacking_ridge = RidgeClassifier(**study.best_params).fit(X_train, y_train.PitNextLap)

# Submission

In [10]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [13]:
submission['PitNextLap'] = stacking_ridge.decision_function(X_test)

In [14]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [15]:
X_train.columns

Index(['lgbm_logit', 'cat_logit', 'xgb_logit', 'hist_logit', 'lg_logit',
       'knn_logit', 'ridge_logit', 'voting_lgbm_cat_xgb_hist_logit',
       'voting_lg_ridge_logit',
       'stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit'],
      dtype='str')